In [2]:
# IMPORTS

from __future__ import annotations
from transformers import AutoTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [7]:
class LLM:
    def __init__(self):
        model_name = "google/flan-t5-base"

        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        print("Loading model...")
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


    def ask(self, prompt, debug=False):
        # (1) Input truncation
        # Prevents crashes or silent errors when prompt gets too long
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256
        )

        # (2) Deterministic generation
        # Removes randomness → same input = same output (important for evaluation)
        # (3) Controlled output length
        # Prevents long rambling responses
        outputs = self.model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.2,
            no_repeat_ngram_size=2
        )

        # (4) Clean decoding
        # Removes special tokens and whitespace noise
        response = self.tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip()

        # (5) Debug mode
        # Lets you inspect exactly what the model saw and returned
        if debug:
            print("\n--- PROMPT ---\n", prompt)
            print("\n--- RESPONSE ---\n", response)

        return response


    def ask_choice(self, prompt, choices):
        # (6) Output constraint via prompt engineering
        # Forces model into a limited decision space
        full_prompt = f"""
{prompt}

Answer ONLY with one of: {", ".join(choices)}.
"""

        response = self.ask(full_prompt)

        # (7) Post-processing / output parsing
        # Converts messy LLM output into clean, usable values
        for c in choices:
            if c.lower() in response.lower():
                return c

        # (8) Fallback handling
        # If model fails, return raw output (helps debugging)
        return response
    
llm = LLM()

Loading tokenizer...
Loading model...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 5968.88it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [10]:
prompt = """
**Rules**
You are a navigation agent operating in a 2D grid-based world.
Your ultimate mission is to reach the "goal tile".
You MUST do so via the most optimal sequence of moves.

Your perceptive field is limited to 3 tiles around you:
- Ahead: the tile directly in front of you
- Left: the tile directly to your left
- Right: the tile directly to your right

**Movement rules:**
- You CANNOT enter wall tiles
- You CAN move onto floor tiles

**Reasoning Protocol — follow each step in order:**
1. Build, in your working memory, a map of the place as you explore it.
2. Assess immediate moves: Which of the valid actions are physically possible right now?
3. Evaluate goal proximity: Which passable move most plausibly advances toward an unknown goal, given no walls are blocking that corridor?
4. Select optimal action: Choose the single action with the best forward progress potential while avoiding immediate dead ends.
5. If all forward paths are blocked, turn to explore a different direction.

Current Environment:
Left = wall
Front = wall
Right = wall

**Output format:**
ONLY reply with either of these three choices:
- [foward]
- [left]
- [right]
"""
response = llm.ask(prompt)
print(response)

Ahead or right


In [ ]:
prompt = """
**Rules**
You are a navigation agent operating in a 2D grid-based world.
Your ultimate mission is to reach the "goal tile".
You MUST do so via the most optimal sequence of moves.

Your perceptive field is limited to 3 tiles around you:
- Ahead: the tile directly in front of you
- Left: the tile directly to your left
- Right: the tile directly to your right

You will be given a memory of previously observed tiles as a map below. 
Each tile is represented by coordinates (x, y) and a tile type. 
All coordinates are relative to the original starting position (0, 0).

**Movement rules:**
- You CANNOT enter wall tiles
- You CAN move onto floor tiles

**Reasoning Protocol — follow each step in order:**
1. Build, in your working memory, a map of the place as you explore it.
2. Assess immediate moves: Which of the valid actions are physically possible right now?
3. Evaluate goal proximity: Which passable move most plausibly advances toward an unknown goal, given no walls are blocking that corridor?
4. Select optimal action: Choose the single action with the best forward progress potential while avoiding immediate dead ends.
5. When the goal tile is not in Current Environment assess from memory of visited tiles and attempt to explore to unknown tiles.

Memory of visited tiles:
(0, 0) = floor
(0, 1) = floor
(-1, 0) = wall
(1, 0) = wall
(0, 2) = unknown
(-1, 1) = unknown
(1, 1) = unknown

Current postion relative to memory map:
(0,1)

Current Environment:
Left = wall
Front = floor
Right = wall

**Output format:**
ONLY reply with either of these three choices:
- [foward]
- [left]
- [right]
"""
response = llm.ask(prompt)
print(response)

4.


In [ ]:
prompt = """
You are a navigation agent in a 2D grid world. Your objective is to reach the "goal tile" using the most optimal sequence of moves.

# Reasoning Protocol

Follow these steps in order to determine the next optimal move:

1.  **Map Construction:** Update your internal map with all observed tiles from the `Memory of visited tiles` and the `Current Environment`.
2.  **Immediate Move Assessment:** Identify all physically possible moves from your `Current position` based on the `Movement rules` and the `Current Environment`.
3.  **Goal Proximity Evaluation:** For each possible move, evaluate which direction most plausibly leads towards the unknown goal, considering potential corridors and avoiding immediate dead ends.
4.  **Optimal Action Selection:** Choose the single action (forward, left, or right) that offers the best potential for forward progress towards the goal while ensuring you do not enter a dead end.
5.  **Exploration:** Prioritize exploring unknown tiles if they appear to be the most optimal path.

# Memory of Visited Tiles

-   (0, 0) = floor
-   (0, 1) = floor
-   (-1, 0) = wall
-   (1, 0) = wall
-   (0, 2) = unknown
-   (-1, 1) = unknown
-   (1, 1) = unknown

# Current State

**Current Position (relative to memory map):** 
(0, 1)

-   **Current Environment:**
    -   Left: wall
    -   Front: floor
    -   Right: wall

# Movement Rules

-   You CANNOT enter wall tiles.
-   You CAN move onto floor tiles.

# Output Format

Respond ONLY with one of the following choices:
-   [foward]
-   [left]
-   [right]
"""
response = llm.ask(prompt)
print(response)

0
